# AST linear-probing — Strategy A

Fine-tunes a linear classifier on top of mean-pooled AST encoder
embeddings. The pretrained AST (HF `MIT/ast-finetuned-audioset-10-10-0.4593`)
is used as a frozen feature extractor.

**Why linear probing, not full fine-tuning:** AST has 86M params; full
fine-tune on Apple MPS would be ~14 hours per epoch at our batch size,
and the AudioSet-pretrained backbone already covers most of our 15
domestic-audio classes. Linear probing isolates the *transfer quality*
of AST features, which is the question we care about: do AudioSet
features cluster our classes linearly?

Pipeline:
  1. (one-time) Extract 768-D embeddings for train/val/test segments.
     Forward-pass only, frozen backbone. Cached to disk.
  2. Train a `Linear(768, 15)` head with 3 imbalance configs.
  3. Per-class threshold tuning on val, test eval once.


## Cell 1 — Setup


In [1]:
# ============================================================================
# Cell 1 — Setup
# ============================================================================
import sys, time, json
from pathlib import Path
import numpy as np
import torch
from torch import nn

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'config.json').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / '_shared'))

from data         import load_splits
from ast_finetune import (load_ast, extract_embeddings,
                          save_embeddings, load_embeddings)
from train_nn     import (auto_device, eval_probs, mean_ap,
                          train_one_run, seed_all)
from imbalance    import compute_class_weights, make_focal_loss
from eval         import (macro_f1, tune_thresholds_on_val,
                          predict_with_threshold,
                          classification_report)

STRATEGY   = 'A'
NB_ID      = '10'
MODEL_NAME = 'ast'
SEED       = 42
DEVICE     = auto_device()
print(f'Device: {DEVICE}')
seed_all(SEED)


Device: mps


## Cell 2 — Load splits and check embedding cache
Embeddings are cached under `strategy_A_baseline/artifacts/ast_embeddings/`.
First run extracts them all (~30-60 min on MPS); subsequent runs skip
straight to training.


In [2]:
# ============================================================================
# Cell 2 — Load splits, set up cache paths
# ============================================================================
splits = load_splits(STRATEGY)
print(splits.summary())
class_names = list(splits.class_names)

EMB_DIR = (PROJECT_ROOT / 'strategy_A_baseline' / 'artifacts'
           / 'ast_embeddings')
EMB_DIR.mkdir(parents=True, exist_ok=True)
PATHS = {s: EMB_DIR / f'{s}.npz' for s in ['train', 'val', 'test']}

for s, p in PATHS.items():
    print(f'  {s}: {p.name}  [{"cached" if p.is_file() else "NEEDS EXTRACTION"}]')


Strategy A
  X_train: (100166, 960)  Y_train: (100166, 15)
  X_val:   (34245, 960)  Y_val:   (34245, 15)
  X_test:  (33828, 960)  Y_test:  (33828, 15)
  D = 960 features, C = 15 classes
  train: train.npz  [cached]
  val: val.npz  [cached]
  test: test.npz  [cached]


## Cell 3 — Extract embeddings (skip if already cached)


In [3]:
# ============================================================================
# Cell 3 — Extract AST embeddings (one-time)
# ============================================================================
if not all(p.is_file() for p in PATHS.values()):
    print('Loading pretrained AST (frozen backbone, head replaced)...')
    model, fe = load_ast(num_labels=15, freeze_backbone=True)
    model = model.to(DEVICE).eval()

    BATCH = 32
    for split_name, idx, Y in [
        ('train', splits.train_idx, splits.Y_train),
        ('val',   splits.val_idx,   splits.Y_val),
        ('test',  splits.test_idx,  splits.Y_test),
    ]:
        path = PATHS[split_name]
        if path.is_file():
            print(f'  [{split_name}] cached, skipping')
            continue
        print(f'  [{split_name}] extracting {len(idx):,} embeddings ...')
        t0 = time.time()
        embs, Yout = extract_embeddings(
            model, fe, idx, Y,
            batch_size=BATCH, device=DEVICE, num_workers=0)
        save_embeddings(path, embs, Yout, idx, split_name)
        print(f'    saved {embs.shape}  ({time.time()-t0:.1f}s, '
              f'{(time.time()-t0)/len(idx)*1000:.1f} ms/sample)')
    del model, fe
    if DEVICE.type == 'mps':
        torch.mps.empty_cache()
else:
    print('All splits cached.')

tr = load_embeddings(PATHS['train'])
va = load_embeddings(PATHS['val'])
te = load_embeddings(PATHS['test'])

X_tr, Y_tr = tr['embeddings'], tr['Y']
X_va, Y_va = va['embeddings'], va['Y']
X_te, Y_te = te['embeddings'], te['Y']
print(f'embeddings: train {X_tr.shape}  val {X_va.shape}  test {X_te.shape}')


All splits cached.
embeddings: train (13889, 768)  val (8851, 768)  test (33828, 768)


## Cell 4 — Linear head + DataLoaders


In [4]:
# ============================================================================
# Cell 4 — Build linear-head DataLoaders
# ============================================================================
from torch.utils.data import TensorDataset, DataLoader

def make_loader(X, Y, batch_size=512, shuffle=False):
    ds = TensorDataset(torch.as_tensor(X, dtype=torch.float32),
                       torch.as_tensor(Y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=0, drop_last=False)

dl_train = make_loader(X_tr, Y_tr, batch_size=512, shuffle=True)
dl_val   = make_loader(X_va, Y_va, batch_size=1024)
dl_test  = make_loader(X_te, Y_te, batch_size=1024)

INPUT_DIM = X_tr.shape[1]
N_CLASSES = int(splits.C)

def fresh_head():
    seed_all(SEED)
    return nn.Sequential(nn.Dropout(0.2),
                         nn.Linear(INPUT_DIM, N_CLASSES))

_h = fresh_head()
print(f'Linear head params: {sum(p.numel() for p in _h.parameters()):,}')
del _h


Linear head params: 11,535


## Cell 5 — Config 1: vanilla BCE


In [5]:
# ============================================================================
# Cell 5 — Config 1: vanilla BCE
# ============================================================================
print('=== Config 1: vanilla BCE ===')
h1 = fresh_head().to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(h1.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)

t0 = time.time()
state1, hist1 = train_one_run(
    h1, dl_train, dl_val, loss_fn, opt,
    scheduler=sched, device=DEVICE,
    max_epochs=30, min_epochs=8, patience=5)
print(f'C1 done in {time.time()-t0:.1f}s | {hist1.summary()}')
h1.load_state_dict(state1)
probs_val_c1, Y_val = eval_probs(h1, dl_val, DEVICE)
print(f'C1 val: mAP={mean_ap(probs_val_c1, Y_val):.4f} '
      f"F1@0.5={macro_f1(Y_val, (probs_val_c1>=0.5).astype(np.int8)):.4f}")


=== Config 1: vanilla BCE ===


  ep  1  tr_loss=0.3255  val_mAP=0.4727  val_F1@0.5=0.0052  skipped=6


  ep  2  tr_loss=0.2298  val_mAP=0.5740  val_F1@0.5=0.1858  skipped=0


  ep  3  tr_loss=0.1948  val_mAP=0.6121  val_F1@0.5=0.3402  skipped=1


  ep  4  tr_loss=0.1785  val_mAP=0.6339  val_F1@0.5=0.4261  skipped=4


...

  ep 26  tr_loss=0.1335  val_mAP=0.7091  val_F1@0.5=0.6097  skipped=13


  ep 27  tr_loss=0.1333  val_mAP=0.7092  val_F1@0.5=0.6099  skipped=9


  ep 28  tr_loss=0.1334  val_mAP=0.7092  val_F1@0.5=0.6103  skipped=4


  ep 29  tr_loss=0.1332  val_mAP=0.7093  val_F1@0.5=0.6113  skipped=8


  ep 30  tr_loss=0.1328  val_mAP=0.7093  val_F1@0.5=0.6111  skipped=5
C1 done in 10.5s | best epoch=30, best val mAP=0.7093, final val F1@0.5=0.6111, final train loss=0.1328
C1 val: mAP=0.7093 F1@0.5=0.6111


## Cell 6 — Config 2: pos-weighted BCE


In [6]:
# ============================================================================
# Cell 6 — Config 2: pos-weighted BCE
# ============================================================================
print('=== Config 2: pos-weighted BCE ===')
pw = compute_class_weights(Y_tr, kind='sqrt_inverse_freq', clip=(0.5, 30.0))
print(f'pos_weight: min={pw.min():.2f}  max={pw.max():.2f}')

h2 = fresh_head().to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.as_tensor(pw, device=DEVICE))
opt = torch.optim.AdamW(h2.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)

t0 = time.time()
state2, hist2 = train_one_run(
    h2, dl_train, dl_val, loss_fn, opt,
    scheduler=sched, device=DEVICE,
    max_epochs=30, min_epochs=8, patience=5)
print(f'C2 done in {time.time()-t0:.1f}s | {hist2.summary()}')
h2.load_state_dict(state2)
probs_val_c2, _ = eval_probs(h2, dl_val, DEVICE)
print(f'C2 val: mAP={mean_ap(probs_val_c2, Y_val):.4f} '
      f"F1@0.5={macro_f1(Y_val, (probs_val_c2>=0.5).astype(np.int8)):.4f}")


=== Config 2: pos-weighted BCE ===
pos_weight: min=2.67  max=7.13


  ep  1  tr_loss=0.5889  val_mAP=0.5564  val_F1@0.5=0.3568  skipped=3


  ep  2  tr_loss=0.4474  val_mAP=0.5941  val_F1@0.5=0.5080  skipped=10


  ep  3  tr_loss=0.3953  val_mAP=0.6275  val_F1@0.5=0.5875  skipped=3


  ep  4  tr_loss=0.3721  val_mAP=0.6382  val_F1@0.5=0.5957  skipped=17


...

  ep 26  tr_loss=0.2817  val_mAP=0.7053  val_F1@0.5=0.6494  skipped=1


  ep 27  tr_loss=0.2805  val_mAP=0.7054  val_F1@0.5=0.6497  skipped=3


  ep 28  tr_loss=0.2809  val_mAP=0.7055  val_F1@0.5=0.6493  skipped=3


  ep 29  tr_loss=0.2802  val_mAP=0.7055  val_F1@0.5=0.6493  skipped=2


  ep 30  tr_loss=0.2807  val_mAP=0.7055  val_F1@0.5=0.6492  skipped=3
C2 done in 7.8s | best epoch=30, best val mAP=0.7055, final val F1@0.5=0.6492, final train loss=0.2807
C2 val: mAP=0.7055 F1@0.5=0.6492


## Cell 7 — Config 3: focal loss γ=2


In [7]:
# ============================================================================
# Cell 7 — Config 3: focal loss γ=2
# ============================================================================
print('=== Config 3: focal loss γ=2 ===')
h3 = fresh_head().to(DEVICE)
loss_fn = make_focal_loss(gamma=2.0, pos_weight=None).to(DEVICE)
opt = torch.optim.AdamW(h3.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)

t0 = time.time()
state3, hist3 = train_one_run(
    h3, dl_train, dl_val, loss_fn, opt,
    scheduler=sched, device=DEVICE,
    max_epochs=30, min_epochs=8, patience=5)
print(f'C3 done in {time.time()-t0:.1f}s | {hist3.summary()}')
h3.load_state_dict(state3)
probs_val_c3, _ = eval_probs(h3, dl_val, DEVICE)
print(f'C3 val: mAP={mean_ap(probs_val_c3, Y_val):.4f} '
      f"F1@0.5={macro_f1(Y_val, (probs_val_c3>=0.5).astype(np.int8)):.4f}")


=== Config 3: focal loss γ=2 ===


  ep  1  tr_loss=0.0877  val_mAP=0.5063  val_F1@0.5=0.1851  skipped=8


  ep  2  tr_loss=0.0548  val_mAP=0.6172  val_F1@0.5=0.4390  skipped=1


  ep  3  tr_loss=0.0462  val_mAP=0.6566  val_F1@0.5=0.5017  skipped=0


  ep  4  tr_loss=0.0431  val_mAP=0.6733  val_F1@0.5=0.5396  skipped=2


...

  ep 26  tr_loss=0.0347  val_mAP=0.7218  val_F1@0.5=0.6360  skipped=4


  ep 27  tr_loss=0.0346  val_mAP=0.7219  val_F1@0.5=0.6385  skipped=2


  ep 28  tr_loss=0.0345  val_mAP=0.7220  val_F1@0.5=0.6370  skipped=5


  ep 29  tr_loss=0.0343  val_mAP=0.7220  val_F1@0.5=0.6373  skipped=3


  ep 30  tr_loss=0.0345  val_mAP=0.7220  val_F1@0.5=0.6372  skipped=6
C3 done in 8.6s | best epoch=30, best val mAP=0.7220, final val F1@0.5=0.6372, final train loss=0.0345
C3 val: mAP=0.7220 F1@0.5=0.6372


## Cell 8 — Best base + per-class thresholds, then test


In [8]:
# ============================================================================
# Cell 8 — Best by val mAP, tune thresholds on val, test ONCE
# ============================================================================
configs = {
    'C1': (probs_val_c1, hist1, h1),
    'C2': (probs_val_c2, hist2, h2),
    'C3': (probs_val_c3, hist3, h3),
}
best_key = max(configs, key=lambda k: mean_ap(configs[k][0], Y_val))
print(f'Best base (val mAP): {best_key}')

probs_val_best, _, model_best = configs[best_key]
thr_df = tune_thresholds_on_val(
    probs_val_best, Y_val,
    grid=np.arange(0.05, 0.96, 0.01),
    class_names=class_names)
print(thr_df.round(3))
thresholds = thr_df['threshold'].values
val_macro_f1_c4 = macro_f1(
    Y_val, predict_with_threshold(probs_val_best, thresholds))
print(f'C4 val macro F1 (tuned): {val_macro_f1_c4:.4f}')

val_macro = {
    'C1_vanilla':          macro_f1(Y_val, (probs_val_c1 >= 0.5).astype(np.int8)),
    'C2_pos_weighted':     macro_f1(Y_val, (probs_val_c2 >= 0.5).astype(np.int8)),
    'C3_focal':            macro_f1(Y_val, (probs_val_c3 >= 0.5).astype(np.int8)),
    f'C4_{best_key}_thr':  val_macro_f1_c4,
}
for k, v in val_macro.items():
    print(f'  {k:25s} {v:.4f}')
winner = max(val_macro, key=val_macro.get)
print(f'\nVal winner: {winner}')

if winner.startswith('C4'):
    probs_test, Y_test = eval_probs(model_best, dl_test, DEVICE)
    Y_pred_test = predict_with_threshold(probs_test, thresholds)
else:
    mdl = {'C1_vanilla': h1, 'C2_pos_weighted': h2, 'C3_focal': h3}[winner]
    probs_test, Y_test = eval_probs(mdl, dl_test, DEVICE)
    Y_pred_test = (probs_test >= 0.5).astype(np.int8)

test_report = classification_report(Y_test, Y_pred_test,
                                   class_names=class_names)
test_macro    = float(test_report.loc['macro avg',    'f1'])
test_micro    = float(test_report.loc['micro avg',    'f1'])
test_weighted = float(test_report.loc['weighted avg', 'f1'])
print(f'\nTEST macro F1:    {test_macro:.4f}')
print(f'TEST micro F1:    {test_micro:.4f}')
print(f'TEST weighted F1: {test_weighted:.4f}')


Best base (val mAP): C3


                            threshold  val_f1  val_precision  val_recall  \
class                                                                      
bell_ringing                     0.38   0.586          0.528       0.660   
coffee_machine                   0.48   0.726          0.877       0.620   
cutlery_dishes                   0.46   0.734          0.796       0.680   
door_open_close                  0.38   0.486          0.438       0.546   
footsteps                        0.40   0.646          0.616       0.680   
keyboard_typing                  0.46   0.805          0.876       0.745   
keychain                         0.44   0.798          0.823       0.776   
light_switch                     0.48   0.521          0.747       0.400   
microwave                        0.41   0.671          0.735       0.617   
phone_ringing                    0.45   0.740          0.811       0.681   
running_water                    0.46   0.768          0.766       0.770   
toilet_flush


TEST macro F1:    0.5859
TEST micro F1:    0.6532
TEST weighted F1: 0.6798


## Cell 9 — Plots


In [9]:
# ============================================================================
# Cell 9 — Plots
# ============================================================================
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

RESULTS = PROJECT_ROOT / 'strategy_A_baseline' / 'artifacts' / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

for cfg, hist in [('c1', hist1), ('c2', hist2), ('c3', hist3)]:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(hist.train_loss); ax[0].set_xlabel('epoch'); ax[0].set_ylabel('train loss')
    ax[0].set_title(f'10 AST {cfg.upper()} — train loss')
    ax[1].plot(hist.val_mAP, label='val mAP')
    ax[1].plot(hist.val_f1_05, label='val F1@0.5')
    ax[1].axvline(hist.best_epoch - 1, ls='--', c='gray', alpha=0.5,
                  label=f'best ep={hist.best_epoch}')
    ax[1].set_xlabel('epoch'); ax[1].legend()
    ax[1].set_title(f'10 AST {cfg.upper()} — val metrics')
    plt.tight_layout()
    plt.savefig(RESULTS / f'10_ast_curves_{cfg}.png', dpi=120, bbox_inches='tight')
    plt.close()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(N_CLASSES), test_report.loc[class_names, 'f1'].values)
ax.set_xticks(range(N_CLASSES)); ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_ylabel('F1 (test)'); ax.set_title(f'10 AST ({winner}) — per-class F1 on test')
plt.tight_layout(); plt.savefig(RESULTS / '10_ast_per_class_f1.png', dpi=120, bbox_inches='tight'); plt.close()

Y_t = Y_test.argmax(axis=1); Y_p = Y_pred_test.argmax(axis=1)
cm = confusion_matrix(Y_t, Y_p, labels=range(N_CLASSES))
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(class_names, rotation=45, ha='right'); ax.set_yticklabels(class_names)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'10 AST ({winner}) — confusion (argmax)')
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout(); plt.savefig(RESULTS / '10_ast_confusion.png', dpi=120, bbox_inches='tight'); plt.close()
print(f'plots -> {RESULTS}')


plots -> /Users/amrabdullah/AI_2026ss/mlpc/strategy_A_baseline/artifacts/results


## Cell 10 — Save results JSON


In [10]:
# ============================================================================
# Cell 10 — Save 10_ast.json
# ============================================================================
result = {
    'meta': {
        'strategy':    STRATEGY,
        'model':       MODEL_NAME,
        'notebook_id': NB_ID,
        'n_train':     int(X_tr.shape[0]),
        'n_val':       int(X_va.shape[0]),
        'n_test':      int(X_te.shape[0]),
        'embedding_dim': int(INPUT_DIM),
        'backbone':    'MIT/ast-finetuned-audioset-10-10-0.4593',
        'mode':        'linear_probing',
        'device':      str(DEVICE),
        'random_seed': SEED,
    },
    'val_scores_macro_f1': val_macro,
    'val_scores_mAP': {
        'C1': float(mean_ap(probs_val_c1, Y_val)),
        'C2': float(mean_ap(probs_val_c2, Y_val)),
        'C3': float(mean_ap(probs_val_c3, Y_val)),
    },
    'histories': {
        c: {'train_loss': h.train_loss, 'val_mAP': h.val_mAP,
            'val_f1_05': h.val_f1_05, 'best_epoch': h.best_epoch}
        for c, h in [('C1', hist1), ('C2', hist2), ('C3', hist3)]
    },
    'thresholds_c4': {c: float(t) for c, t in zip(class_names, thresholds)},
    'winning_config': {
        'name': winner,
        'thresholds': [float(t) for t in thresholds]
                      if winner.startswith('C4')
                      else [0.5] * N_CLASSES,
    },
    'test_report': test_report.to_dict(),
    'headline_test': {
        'macro_f1':    test_macro,
        'micro_f1':    test_micro,
        'weighted_f1': test_weighted,
    },
}

out_path = RESULTS / f'{NB_ID}_{MODEL_NAME}.json'
with open(out_path, 'w') as f:
    json.dump(result, f, indent=2, default=float)
print(f'saved -> {out_path}')


saved -> /Users/amrabdullah/AI_2026ss/mlpc/strategy_A_baseline/artifacts/results/10_ast.json
